# TFLite로 경량화


In [7]:
! pip install tensorflow

   ---------------------------------------- 0.0/390.0 MB ? eta -:--:--
   ---------------------------------------- 2.1/390.0 MB 11.8 MB/s eta 0:00:33
   ---------------------------------------- 4.7/390.0 MB 11.4 MB/s eta 0:00:34
    --------------------------------------- 6.8/390.0 MB 11.3 MB/s eta 0:00:34
    --------------------------------------- 9.4/390.0 MB 11.5 MB/s eta 0:00:34
   - -------------------------------------- 12.1/390.0 MB 11.4 MB/s eta 0:00:34
   - -------------------------------------- 14.4/390.0 MB 11.5 MB/s eta 0:00:33
   - -------------------------------------- 17.0/390.0 MB 11.5 MB/s eta 0:00:33
   - -------------------------------------- 19.4/390.0 MB 11.4 MB/s eta 0:00:33
   -- ------------------------------------- 21.8/390.0 MB 11.5 MB/s eta 0:00:33
   -- ------------------------------------- 24.1/390.0 MB 11.5 MB/s eta 0:00:32
   -- ------------------------------------- 26.7/390.0 MB 11.5 MB/s eta 0:00:32
   -- ------------------------------------- 29.1/390.

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imgaug 0.4.0 requires matplotlib, which is not installed.
mlflow 2.11.0 requires matplotlib<4, which is not installed.
pixellib 0.7.1 requires matplotlib, which is not installed.
seaborn 0.13.2 requires matplotlib!=3.6.1,>=3.4, which is not installed.
datasets 3.2.0 requires requests>=2.32.2, but you have requests 2.25.1 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
mlflow 2.11.0 requires numpy<2, but you have numpy 2.0.2 which is incompatible.
pyarrow 15.0.2 requires numpy<2,>=1.16.6, but you have numpy 2.0.2 which is incompatible.
sahi 0.11.20 requires numpy<2.0.0, but you have numpy 2.0.2 which is incompatible.


In [1]:
import tensorflow as tf

In [2]:
import os

In [3]:
! pip install keras

In [6]:
import keras

ModuleNotFoundError: No module named 'tensorflow.python'

## 1. 저장된 모델 로드

In [5]:
load_path = r"C:\Users\User\Documents\GitHub\mlops\7_MLOps\cifar10_tuned_model\1"
best_model = tf.keras.models.load_model(load_path)

NameError: name 'keras' is not defined

In [13]:
model_path = './cifar10_tuned_model/1'

# 저장된 모델 로드 (Load saved model)
try:
    model = tf.keras.models.load_model(model_path)
    print("모델 로드 완료 (Model loaded successfully)")
    print(f"\n모델 구조 (Model structure):")
    model.summary()
except Exception as e:
    print(f"모델 로드 실패 (Model loading failed): {str(e)}")
    print(f"\n현재 디렉토리 내용 (Current directory contents):")
    print(os.listdir('.'))

모델 로드 실패 (Model loading failed): 'utf-8' codec can't decode byte 0xbc in position 64: invalid start byte

현재 디렉토리 내용 (Current directory contents):
['7_MLOps', 'assets', 'best_model', 'check_model.py', 'check_original.py', 'check_training.py', 'cifar10_tuned_model', 'colab_env.yaml', 'convert_to_tflite.ipynb', 'convert_to_tflite.py', 'environment.yaml', 'keras_tuner.ipynb', 'keras_tuner_cifar10_project.ipynb', 'keras_tuner_colab.ipynb', 'my_keras_tuner', 'quick_test.py', 'save_all.py', 'simple_test.py', 'test_all.py', 'test_model', 'test_model_serving.ipynb', 'test_serving.py', 'train_test.py']


## 2. TFLite 모델로 변환

In [11]:
# 1. 먼저 모델을 제대로 로드
try:
    model = tf.keras.models.load_model('./cifar10_tuned_model/1')
    print("모델 로드 완료")
except Exception as e:
    print(f"모델 로드 실패: {str(e)}")
    
# 2. 그 다음 TFLite 변환
if 'model' in locals():  # 모델이 성공적으로 로드된 경우에만 실행
    # TFLite 변환기 생성
    converter = tf.lite.TFLiteConverter.from_keras_model(model)  # 로드된 model 객체 사용
    
    # 최적화 옵션 설정
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    
    # 모델 변환
    tflite_model = converter.convert()
    
    # TFLite 모델 저장
    with open('cifar10_model.tflite', 'wb') as f:
        f.write(tflite_model)
    
    print(f"TFLite 모델 크기: {len(tflite_model) / 1024:.2f} KB")

모델 로드 실패: 'utf-8' codec can't decode byte 0xbc in position 64: invalid start byte


In [ ]:
# # TFLite 변환기 생성 (Create TFLite converter)
# converter = tf.lite.TFLiteConverter.from_keras_model('cifar10_tuned_model/1')

# # 최적화 옵션 설정 (Set optimization options)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.target_spec.supported_types = [tf.float16]

# # 모델 변환 (Convert model)
# tflite_model = converter.convert()

# # TFLite 모델 저장 (Save TFLite model)
# with open('cifar10_model.tflite', 'wb') as f:
#     f.write(tflite_model)

# print(f"TFLite 모델 크기 (TFLite model size): {len(tflite_model) / 1024:.2f} KB")

AttributeError: 'str' object has no attribute 'call'

## 3. 모델 서명 확인

In [ ]:
# TFLite 인터프리터 생성 (Create TFLite interpreter)
interpreter = tf.lite.Interpreter(model_path='cifar10_model.tflite')
interpreter.allocate_tensors()

# 입력과 출력 상세정보 가져오기 (Get input and output details)
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\n입력 텐서 정보 (Input tensor details):")
for detail in input_details:
    print(f"\n이름 (Name): {detail['name']}")
    print(f"형태 (Shape): {detail['shape']}")
    print(f"타입 (Type): {detail['dtype']}")

print("\n출력 텐서 정보 (Output tensor details):")
for detail in output_details:
    print(f"\n이름 (Name): {detail['name']}")
    print(f"형태 (Shape): {detail['shape']}")
    print(f"타입 (Type): {detail['dtype']}")

## 4. 간단한 추론 테스트

In [ ]:
import numpy as np

# 테스트 이미지 생성 (Create test image)
test_image = np.random.rand(1, 32, 32, 3).astype(np.float32)

# 입력 텐서 설정 (Set input tensor)
interpreter.set_tensor(input_details[0]['index'], test_image)

# 추론 실행 (Run inference)
interpreter.invoke()

# 출력 가져오기 (Get output)
output_data = interpreter.get_tensor(output_details[0]['index'])

# 결과 출력 (Print results)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

predicted_class = np.argmax(output_data[0])
print(f"\n예측된 클래스 (Predicted class): {class_names[predicted_class]}")
print("\n각 클래스별 확률 (Probabilities for each class):")
for i, (name, prob) in enumerate(zip(class_names, output_data[0])):
    print(f"{name}: {prob:.4f}")